# Attribution Explanations Visualization

This notebook visualizes pre-computed token/word-level attributions.

Attributions are computed via interpreto methods (saliency, LIME, integrated gradients, etc.)
and cached in `data/{model_dir}/attributions/{method}.pt`.

Each attribution maps tokens/words to importance scores for the predicted class.

## Parameters

In [ ]:
# ---------------------------------------------------------------------------
# Configuration: adjust these to select the dataset / method
# ---------------------------------------------------------------------------

# Dataset short name: "RT", "GE", "BIOS", "AG", "IMDB", "E", "HE"
DATASET_ABBREV = "RT"

# Attribution method: "saliency", "integrated_gradients", "smooth_grad",
#   "square_grad", "var_grad", "gradient_shap", "lime", "kernel_shap", "occlusion", "sobol"
METHOD = "saliency"

# Classes subset index (within DATASET_CLASSES_SUBSETS for the dataset)
CLASSES_SUBSET_IDX = 0

# Seed for sample selection
SEED = 0

# Number of samples per seed
NB_SAMPLES = 5

## Imports and path resolution

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so we can import utils
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from utils.data import (
    MODELS_DATASETS,
    ABBREVIATIONS,
    DATASET_CLASSES_NAMES,
    DATASET_CLASSES_SUBSETS,
    get_save_root,
)
from utils.attributions import ATTRIBUTION_METHODS

In [ ]:
# Resolve full dataset/model names from abbreviation
dataset_name = next(k for k, v in ABBREVIATIONS["datasets"].items() if v == DATASET_ABBREV)
model_name = next(k for k, v in MODELS_DATASETS.items() if v == dataset_name)

# Classes subset
classes_subset = DATASET_CLASSES_SUBSETS[dataset_name][CLASSES_SUBSET_IDX]
classes_names = [DATASET_CLASSES_NAMES[dataset_name][i] for i in classes_subset]

# Paths
save_root = REPO_ROOT / get_save_root(model_name)
attribution_path = save_root / "attributions" / f"{METHOD}.pt"

print(f"Dataset: {dataset_name}")
print(f"Model: {model_name}")
print(f"Attribution method: {METHOD}")
print(f"Classes: {classes_names} (indices {classes_subset})")
print(f"Attribution file: {attribution_path}")
print(f"Exists: {attribution_path.exists()}")

## Load attributions

In [ ]:
# Load cached attributions: dict[int, dict] mapping sample_id -> {attributions, elements, target}
if attribution_path.exists():
    cached_attributions = torch.load(attribution_path, map_location="cpu")
    print(f"Loaded {len(cached_attributions)} cached attributions")
    # Show a sample record structure
    sample_id = next(iter(cached_attributions))
    record = cached_attributions[sample_id]
    print(f"\nExample record (sample_id={sample_id}):")
    print(f"  attributions shape: {record['attributions'].shape}")
    print(f"  elements: {record['elements'][:10]}{'...' if len(record['elements']) > 10 else ''}")
    print(f"  target: {record['target']}")
else:
    cached_attributions = {}
    print(f"WARNING: Attribution file not found at {attribution_path}")
    print(f"Run `python scripts/make_prompts.py attributions {DATASET_ABBREV} {METHOD}` to compute them.")
    print(f"\nAvailable methods: {list(ATTRIBUTION_METHODS.keys())}")
    # Check what attribution files exist
    attr_dir = save_root / "attributions"
    if attr_dir.exists():
        existing = list(attr_dir.glob("*.pt"))
        print(f"\nExisting attribution files: {[f.stem for f in existing]}")

## Load sample selection

In [ ]:
# Load sample selection for this seed
classes_str = "-".join(str(c) for c in classes_subset)
local_elements_path = save_root / f"local_elements_classes_{classes_str}_n{NB_SAMPLES}.json"

if local_elements_path.exists():
    with open(local_elements_path) as f:
        local_elements = json.load(f)
    seed_data = local_elements[str(SEED)]
    sample_indices = seed_data["indices"]
    sample_texts = seed_data["texts"]
    sample_predictions = seed_data["predictions"]
    sample_labels = seed_data["labels"]
    print(f"Seed {SEED}: {len(sample_indices)} samples")
else:
    print(f"WARNING: Local elements file not found: {local_elements_path}")
    # Fallback: use the first N cached sample_ids
    sample_indices = sorted(cached_attributions.keys())[:NB_SAMPLES]
    sample_texts = None
    sample_predictions = None
    sample_labels = None
    print(f"Falling back to first {len(sample_indices)} cached sample_ids")

## Visualize attributions with interpreto

In [ ]:
from interpreto import plot_attributions
from interpreto.attributions.base import AttributionOutput

# Reconstruct AttributionOutput objects for the selected samples
for i, idx in enumerate(sample_indices):
    if idx not in cached_attributions:
        print(f"Sample {i} (idx={idx}): not in cache, skipping.")
        continue

    record = cached_attributions[idx]
    target = record["target"]
    if isinstance(target, int):
        targets = torch.tensor([target])
    else:
        targets = torch.tensor(target)

    attr_output = AttributionOutput(
        attributions=record["attributions"],
        elements=record["elements"],
        model_inputs_to_explain={},
        targets=targets,
        model_task="classification",
    )

    pred_name = DATASET_CLASSES_NAMES[dataset_name][target] if isinstance(target, int) else str(target)
    text_preview = sample_texts[i][:80] if sample_texts else f"sample_id={idx}"
    print(f"\n--- Sample {i} (idx={idx}) | Predicted: {pred_name} ---")
    print(f"    \"{text_preview}...\"")

    plot_attributions(attr_output, classes_names=classes_names)

## Custom attribution heatmap

A more compact visualization showing token-level attributions as colored text.

In [ ]:
from IPython.display import display, HTML

def render_attribution_html(elements: list[str], attributions: torch.Tensor, title: str = "") -> str:
    """Render tokens with background color proportional to attribution magnitude."""
    # Normalize attributions to [0, 1] for coloring
    attr = attributions.float()
    if attr.ndim > 1:
        # If multi-class attributions, take the max across classes
        attr = attr.abs().max(dim=0).values if attr.shape[0] <= 10 else attr[0]

    abs_attr = attr.abs()
    max_val = abs_attr.max().item() if abs_attr.max().item() > 0 else 1.0

    html = f'<div style="margin:10px 0;"><b>{title}</b><br><p style="line-height:2.2;">'
    for token, score in zip(elements, attr.tolist()):
        # Red for positive, blue for negative
        intensity = abs(score) / max_val
        if score >= 0:
            r, g, b = 255, int(255 * (1 - intensity)), int(255 * (1 - intensity))
        else:
            r, g, b = int(255 * (1 - intensity)), int(255 * (1 - intensity)), 255
        html += f'<span style="background-color:rgb({r},{g},{b}); padding:2px 4px; margin:1px; border-radius:3px;">{token}</span> '
    html += '</p></div>'
    return html


html_parts = []
for i, idx in enumerate(sample_indices):
    if idx not in cached_attributions:
        continue
    record = cached_attributions[idx]
    target = record["target"]
    pred_name = DATASET_CLASSES_NAMES[dataset_name][target] if isinstance(target, int) else str(target)
    label_name = DATASET_CLASSES_NAMES[dataset_name][sample_labels[i]] if sample_labels else "?"

    title = f"Sample {i} (idx={idx}) — Label: {label_name}, Pred: {pred_name}"
    html_parts.append(render_attribution_html(record["elements"], record["attributions"], title))

if html_parts:
    display(HTML("".join(html_parts)))
else:
    print("No attributions available to display.")

## Attribution statistics

In [ ]:
if cached_attributions:
    # Collect statistics over all cached attributions
    all_lengths = []
    all_sparsities = []
    all_max_vals = []

    for record in cached_attributions.values():
        attr = record["attributions"].float()
        if attr.ndim > 1:
            attr = attr.abs().max(dim=0).values
        all_lengths.append(len(record["elements"]))
        # Sparsity: fraction of near-zero attributions
        threshold = 0.01 * attr.abs().max().item() if attr.abs().max().item() > 0 else 0.01
        all_sparsities.append((attr.abs() < threshold).float().mean().item())
        all_max_vals.append(attr.abs().max().item())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(all_lengths, bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Number of tokens/words")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title(f"Input length distribution (n={len(cached_attributions)})")

    axes[1].hist(all_sparsities, bins=30, edgecolor="black", alpha=0.7, color="orange")
    axes[1].set_xlabel("Sparsity (fraction near-zero)")
    axes[1].set_ylabel("Frequency")
    axes[1].set_title(f"Attribution sparsity ({METHOD})")

    axes[2].hist(all_max_vals, bins=30, edgecolor="black", alpha=0.7, color="green")
    axes[2].set_xlabel("Max |attribution|")
    axes[2].set_ylabel("Frequency")
    axes[2].set_title(f"Peak attribution magnitude ({METHOD})")

    plt.tight_layout()
    plt.show()

    print(f"\nTotal cached attributions: {len(cached_attributions)}")
    print(f"Input lengths — min: {min(all_lengths)}, max: {max(all_lengths)}, mean: {np.mean(all_lengths):.1f}")
    print(f"Sparsity — mean: {np.mean(all_sparsities):.3f}")
    print(f"Max |attr| — mean: {np.mean(all_max_vals):.4f}")
else:
    print("No attributions loaded — skipping statistics.")

## Top-K attribution summary per sample

In [ ]:
TOP_K = 6  # number of top tokens to show (matches the default in attrsim.py)

print(f"Top-{TOP_K} attributed tokens per sample:\n")
for i, idx in enumerate(sample_indices):
    if idx not in cached_attributions:
        continue
    record = cached_attributions[idx]
    attr = record["attributions"].float()
    elements = record["elements"]
    target = record["target"]
    pred_name = DATASET_CLASSES_NAMES[dataset_name][target] if isinstance(target, int) else str(target)

    # For multi-dim attributions, take the predicted class row
    if attr.ndim > 1:
        target_idx = target if isinstance(target, int) else target[0]
        attr = attr[target_idx]

    # L1 normalize
    attr_norm = attr / (attr.abs().sum() + 1e-10)

    # Top-k by absolute value
    top_indices = torch.argsort(attr_norm.abs(), descending=True)[:TOP_K]

    text_preview = sample_texts[i][:60] if sample_texts else f"id={idx}"
    print(f"Sample {i} (idx={idx}) | Pred: {pred_name} | \"{text_preview}...\"")
    for j, tok_idx in enumerate(top_indices):
        token = elements[tok_idx]
        score = attr_norm[tok_idx].item()
        print(f"    {j+1}. {token!r:>15}: {score:+.4f}")
    print()